In [ ]:
# -*- coding: utf-8 -*-
"""
Boston Silver Line (BRT) Criticality Analysis
Adapted from the Istanbul Metrobüs substitute-network methodology.

KEY DIFFERENCE vs. the Istanbul version:
Boston's stop_times.txt has real arrival_time / departure_time, so trunk and
substitute edge weights are ACTUAL SCHEDULED TRAVEL TIME (seconds), not a
haversine-distance proxy. Only the walking-transfer edges (between a Silver
Line stop and an ordinary bus stop) still rely on haversine distance, since
GTFS has no "walking schedule" -- that distance is converted to seconds via
an assumed walking speed so all edge weights share the same units (seconds).

Station consolidation uses GTFS's own parent_station hierarchy (Boston
provides this explicitly) instead of the Istanbul script's regex-based
name normalization, which is more reliable when it's available.
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# CONFIG -- adjust to your environment
# ---------------------------------------------------------------------------

DATA_DIR = Path("/content/drive/MyDrive/Boston GTFS")  # <-- update as needed

FILES = {
    "routes":     DATA_DIR / "routes.txt",
    "trips":      DATA_DIR / "trips.txt",
    "stop_times": DATA_DIR / "stop_times.txt",
    "stops":      DATA_DIR / "stops.txt",
}

IST_LAT_RANGE = (42.0, 42.6)   # Boston-area bounding box (sanity filter)
IST_LON_RANGE = (-71.3, -70.7)

WALKING_SPEED_MPS = 1.4
NO_BACKUP_PENALTY_SEC = 3 * 3600  # "no reasonable backup" = treat as 3 hours

WALK_SCENARIOS = {
    "20_min": 1200,
    "40_min": 2400,
    "60_min": 3600,
}

# ---------------------------------------------------------------------------
# WALKING SCENARIOS – now include very short walksheds
# ---------------------------------------------------------------------------
WALK_SCENARIOS = {
    "1_min":  60,   # 2  minutes → seconds
    "2_min":  120,   # 5  minutes → seconds
    "3_min": 180,   # 10 minutes → seconds
    "4_min": 240,  # keep the original longer ones for comparison
    "5_min": 300,
    "6_min": 360,
}
# ← END OF MODIFIED BLOCK

# Silver Line route codes -- change this if your "trunk" BRT definition differs
TRUNK_ROUTE_PATTERN = r'^SL[0-9]$'

# --- FIX 1: split Silver Line into its structurally different segments ---
# SL1/SL2/(SL3) run through the dedicated South Boston Piers Transitway/Seaport
# tunnel with real separated stations -- structurally like Istanbul's Metrobüs.
# SL4/SL5 run in mixed street traffic on Washington St, often sharing curb
# space with local buses -- backups may already exist at ~0 walking distance,
# which flattens walk-time sensitivity. Analyze these separately to see if
# that's what's happening. Set TRUNK_SUBSET to "all", "tunnel", or "street".
TRUNK_SUBSET = "tunnel"  # <-- change to "tunnel" or "street" to isolate a segment

TRUNK_GROUPS = {
    "tunnel": {"SL1", "SL2", "SL3"},
    "street": {"SL4", "SL5"},
    "all":    {"SL1", "SL2", "SL3", "SL4", "SL5"},
}

# --- FIX 2: fallback consolidation for on-street stops lacking parent_station ---
# Two directional platforms of the same physical street stop (e.g. "Washington
# St @ Lenox St" northbound/southbound) often have no parent_station in GTFS.
# Stops that share a normalized name AND are within this radius get merged
# into a single station node, same as the Istanbul name-normalization step,
# but only applied as a fallback where parent_station didn't already handle it.
CONSOLIDATION_PROXIMITY_M = 150

pd.set_option("display.max_columns", None)
print(f"Walking speed assumption: {WALKING_SPEED_MPS} m/s")
print("Environment ready.")

# ---------------------------------------------------------------------------
# LOADING
# ---------------------------------------------------------------------------

REQUIRED_COLS = {
    "routes":     ["route_id", "route_short_name"],
    "trips":      ["trip_id", "route_id"],
    "stop_times": ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
    "stops":      ["stop_id", "stop_name", "stop_lat", "stop_lon"],
}

def load_gtfs_file(name, path, required_cols, separator=','):
    df = None
    used_encoding = None
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            df = pd.read_csv(
                path, encoding=enc, dtype=str, sep=separator,
                skip_blank_lines=True, on_bad_lines="skip", engine="python"
            )
            used_encoding = enc
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise RuntimeError(f"Could not decode {path}.")

    df.columns = df.columns.str.strip()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].str.strip()
    df = df.replace("", np.nan).dropna(how="all").drop_duplicates()
    df = df.dropna(subset=required_cols).reset_index(drop=True)

    print(f"{name:12s} enc={used_encoding:9s} clean={len(df):>7,} rows")
    return df

routes     = load_gtfs_file("routes",     FILES["routes"],     REQUIRED_COLS["routes"])
trips      = load_gtfs_file("trips",      FILES["trips"],      REQUIRED_COLS["trips"])
stop_times = load_gtfs_file("stop_times", FILES["stop_times"], REQUIRED_COLS["stop_times"])
stops      = load_gtfs_file("stops",      FILES["stops"],      REQUIRED_COLS["stops"])

for df_obj in [routes, trips, stop_times, stops]:
    for col in ["route_id", "trip_id", "stop_id", "parent_station"]:
        if col in df_obj.columns:
            df_obj[col] = df_obj[col].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# ---------------------------------------------------------------------------
# TRUNK vs. SUBSTITUTE ROUTE IDENTIFICATION
# ---------------------------------------------------------------------------

routes["route_short_name_clean"] = routes["route_short_name"].astype(str).str.strip().str.upper()
is_silver_line = routes["route_short_name_clean"].str.contains(TRUNK_ROUTE_PATTERN, regex=True)

allowed_codes = TRUNK_GROUPS[TRUNK_SUBSET]
is_trunk = is_silver_line & routes["route_short_name_clean"].isin(allowed_codes)

print(f"\nTRUNK_SUBSET = '{TRUNK_SUBSET}' -> allowed codes: {sorted(allowed_codes)}")
print(f"Silver Line routes present in this feed: {sorted(routes.loc[is_silver_line, 'route_short_name_clean'].unique().tolist())}")
print(f"Trunk routes actually used: {sorted(routes.loc[is_trunk, 'route_short_name_clean'].unique().tolist())}")

# Everything that is NOT selected as trunk (including the OTHER Silver Line
# branches when TRUNK_SUBSET is "tunnel" or "street") counts as substitute --
# this lets e.g. SL4/SL5 act as a real backup option for SL1/SL2 riders.
trunk_trip_ids = set(trips.loc[trips["route_id"].isin(routes[is_trunk]["route_id"]), "trip_id"])
substitute_trip_ids = set(trips.loc[trips["route_id"].isin(routes[~is_trunk]["route_id"]), "trip_id"])

# ---------------------------------------------------------------------------
# STATION CONSOLIDATION: parent_station first, then a proximity fallback
# ---------------------------------------------------------------------------

def build_node_mapping(stops_df):
    """Map every stop_id to an 'effective node id' using parent_station when
    available. This collapses separate platforms/berths into one station
    node, similar in purpose to the Istanbul script's name-normalization
    step, but using GTFS's own hierarchy instead of regex matching."""
    stop_id_set = set(stops_df["stop_id"])
    has_parent = stops_df.get("parent_station", pd.Series(dtype=str)).fillna("")
    effective_id = np.where(
        (has_parent != "") & has_parent.isin(stop_id_set),
        has_parent,
        stops_df["stop_id"],
    )
    mapping = dict(zip(stops_df["stop_id"], effective_id))
    return mapping

node_id_map = build_node_mapping(stops)
stops["node_id"] = stops["stop_id"].map(node_id_map)

def calculate_haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    a = np.sin(np.radians(lat2 - lat1) / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def normalize_name_for_clustering(name):
    """Loose name key used only to group candidate stops for the proximity
    fallback below -- not used as the final node identity."""
    if pd.isna(name):
        return name
    name = str(name).strip().upper()
    name = re.sub(r'\s*-\s*(INBOUND|OUTBOUND|EB|WB|NB|SB|SILVER LINE)\s*$', '', name)
    return name

def apply_proximity_fallback(stops_df, node_id_map, proximity_m):
    """FIX 2: for stops that did NOT get merged via parent_station (i.e. their
    node_id is still just their own stop_id), merge stops that share a
    normalized name AND sit within `proximity_m` meters of each other -- e.g.
    the two directional platforms of an on-street stop like "Washington St @
    Lenox St" that GTFS never links via parent_station."""
    df = stops_df.copy()
    df["node_id"] = df["stop_id"].map(node_id_map)
    df["lat_f"] = pd.to_numeric(df["stop_lat"], errors="coerce")
    df["lon_f"] = pd.to_numeric(df["stop_lon"], errors="coerce")
    df = df.dropna(subset=["lat_f", "lon_f"])

    # Only candidates for merging: stops still mapped to themselves (i.e. no
    # parent_station consolidation already happened for them).
    unconsolidated = df[df["node_id"] == df["stop_id"]].copy()
    unconsolidated["name_key"] = unconsolidated["stop_name"].apply(normalize_name_for_clustering)

    updated_map = dict(node_id_map)
    merges_made = 0

    for name_key, group in unconsolidated.groupby("name_key"):
        if len(group) < 2:
            continue
        ids = group["stop_id"].tolist()
        lats = group["lat_f"].values
        lons = group["lon_f"].values

        # simple union-find over pairwise haversine distance within the group
        parent = {i: i for i in ids}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb

        n = len(ids)
        for i in range(n):
            for j in range(i + 1, n):
                d = calculate_haversine(lats[i], lons[i], lats[j], lons[j])
                if d <= proximity_m:
                    union(ids[i], ids[j])

        clusters = {}
        for stop_id in ids:
            root = find(stop_id)
            clusters.setdefault(root, []).append(stop_id)

        for members in clusters.values():
            if len(members) < 2:
                continue
            canonical = sorted(members)[0]
            for m in members:
                updated_map[m] = canonical
            merges_made += 1

    print(f"Proximity fallback: merged {merges_made} groups of on-street stops "
          f"(name match + within {proximity_m}m) that parent_station missed.")
    return updated_map

node_id_map = apply_proximity_fallback(stops, node_id_map, CONSOLIDATION_PROXIMITY_M)
stops["node_id"] = stops["stop_id"].map(node_id_map)

# display name for a node = the parent station's name if it exists, else own name
id_to_name = dict(zip(stops["stop_id"], stops["stop_name"]))
node_name_map = {}
for node_id in stops["node_id"].unique():
    node_name_map[node_id] = id_to_name.get(node_id, node_id)

# ---------------------------------------------------------------------------
# COORDINATES (consolidated to node level, mean of constituent stops)
# ---------------------------------------------------------------------------

coords = stops[["stop_id", "node_id", "stop_lat", "stop_lon"]].copy()
coords["lat_f"] = pd.to_numeric(coords["stop_lat"], errors="coerce")
coords["lon_f"] = pd.to_numeric(coords["stop_lon"], errors="coerce")

out_of_bounds = ~coords["lat_f"].between(*IST_LAT_RANGE) | ~coords["lon_f"].between(*IST_LON_RANGE)
coords.loc[out_of_bounds, ["lat_f", "lon_f"]] = np.nan
coords = coords.dropna(subset=["lat_f", "lon_f"])

consolidated_coords = coords.groupby("node_id")[["lat_f", "lon_f"]].mean().reset_index()
consolidated_coords["node_name"] = consolidated_coords["node_id"].map(node_name_map)

# ---------------------------------------------------------------------------
# GTFS TIME PARSING (handles times past 24:00:00 for next-day service)
# ---------------------------------------------------------------------------

def gtfs_time_to_seconds(t):
    if pd.isna(t):
        return np.nan
    try:
        h, m, s = str(t).strip().split(":")
        return int(h) * 3600 + int(m) * 60 + int(s)
    except (ValueError, AttributeError):
        return np.nan

stop_times["arr_sec"] = stop_times["arrival_time"].apply(gtfs_time_to_seconds)
stop_times["dep_sec"] = stop_times["departure_time"].apply(gtfs_time_to_seconds)
stop_times = stop_times.dropna(subset=["arr_sec", "dep_sec"])
stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")
stop_times = stop_times.dropna(subset=["stop_sequence"])
stop_times["stop_sequence"] = stop_times["stop_sequence"].astype(int)
stop_times["node_id"] = stop_times["stop_id"].map(node_id_map)
stop_times = stop_times.dropna(subset=["node_id"])

# ---------------------------------------------------------------------------
# BUILD SEGMENT TRAVEL TIMES FROM ACTUAL SCHEDULES
# ---------------------------------------------------------------------------

def build_segments_time(trip_id_set, stop_times_df, label=""):
    df_sub = stop_times_df[stop_times_df["trip_id"].isin(trip_id_set)].copy()
    df_sub = df_sub.sort_values(["trip_id", "stop_sequence"])

    df_sub["next_node"] = df_sub.groupby("trip_id")["node_id"].shift(-1)
    df_sub["next_arr_sec"] = df_sub.groupby("trip_id")["arr_sec"].shift(-1)

    seg = df_sub.dropna(subset=["next_node", "next_arr_sec"]).copy()
    seg = seg[seg["node_id"] != seg["next_node"]]

    # travel time = arrival at next stop - departure from current stop
    seg["travel_sec"] = seg["next_arr_sec"] - seg["dep_sec"]
    seg = seg[seg["travel_sec"] > 0]  # drop bad/overlapping records

    seg = seg.rename(columns={"node_id": "from_id", "next_node": "to_id"})
    print(f"{label}: {len(seg):,} valid segment observations")
    return seg[["from_id", "to_id", "travel_sec"]]

trunk_segments = build_segments_time(trunk_trip_ids, stop_times, "TRUNK (Silver Line)")
substitute_segments = build_segments_time(substitute_trip_ids, stop_times, "SUBSTITUTE (all other routes)")

def build_weighted_graph(segments_df):
    edge_data = segments_df.groupby(["from_id", "to_id"])["travel_sec"].median().reset_index()
    G = nx.Graph()
    for _, row in edge_data.iterrows():
        u, v, w = row["from_id"], row["to_id"], row["travel_sec"]
        if G.has_edge(u, v):
            G[u][v]["weight"] = min(G[u][v]["weight"], w)
        else:
            G.add_edge(u, v, weight=w)
    return G

G_trunk = build_weighted_graph(trunk_segments)
G_sub = build_weighted_graph(substitute_segments)
print(f"\nG_trunk: {G_trunk.number_of_nodes()} nodes, {G_trunk.number_of_edges()} edges (weights = median scheduled seconds)")
print(f"G_sub:   {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")

# ---------------------------------------------------------------------------
# FIX 3 -- DIAGNOSTIC: how many trunk stations already sit directly on the
# substitute network (i.e. a local bus stops at/near the same physical node),
# meaning a "backup" exists at essentially zero walking distance regardless
# of the walkshed scenario? This directly tests the hypothesis that flat
# walk-time sensitivity comes from shared street infrastructure rather than
# a script bug.
# ---------------------------------------------------------------------------

trunk_nodes_set = set(G_trunk.nodes)
sub_nodes_set = set(G_sub.nodes)
zero_distance_backup_nodes = trunk_nodes_set & sub_nodes_set

print(f"\n[DIAGNOSTIC] Trunk stations that are ALSO directly a substitute-network "
      f"node (same stop_id/parent_station shared with a local route):")
print(f"  {len(zero_distance_backup_nodes)} of {len(trunk_nodes_set)} trunk stations "
      f"({100*len(zero_distance_backup_nodes)/max(1,len(trunk_nodes_set)):.1f}%)")
if zero_distance_backup_nodes:
    sample_names = [node_name_map.get(n, n) for n in list(zero_distance_backup_nodes)[:10]]
    print(f"  Example stations with a zero-distance backup: {sample_names}")
print("  A high percentage here explains why expanding the walkshed (10->30 min) "
      "may not change rankings: the shortest backup path was already available "
      "without needing any walking-transfer edge at all.")

# ---------------------------------------------------------------------------
# WALKING-TRANSFER EDGES (distance -> time, since no GTFS "walk schedule" exists)
# ---------------------------------------------------------------------------

def build_walking_edges(max_walk_time_sec):
    max_dist_m = WALKING_SPEED_MPS * max_walk_time_sec
    trunk_nodes, sub_nodes = list(G_trunk.nodes), list(G_sub.nodes)

    tc = consolidated_coords.set_index("node_id").reindex(trunk_nodes).dropna()
    sc = consolidated_coords.set_index("node_id").reindex(sub_nodes).dropna()

    lat1, lon1 = np.radians(tc["lat_f"].values)[:, None], np.radians(tc["lon_f"].values)[:, None]
    lat2, lon2 = np.radians(sc["lat_f"].values)[None, :], np.radians(sc["lon_f"].values)[None, :]

    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    dist_matrix = 2 * 6371000 * np.arcsin(np.sqrt(a))

    t_idx, s_idx = np.where(dist_matrix <= max_dist_m)
    walk_time_sec = dist_matrix[t_idx, s_idx] / WALKING_SPEED_MPS

    return pd.DataFrame({
        "trunk_stop": np.array(tc.index)[t_idx],
        "sub_stop":   np.array(sc.index)[s_idx],
        "walk_sec":   walk_time_sec,
    })

# ---------------------------------------------------------------------------
# CRITICALITY COMPUTATION (all weights in seconds now, so L_i is a true
# "how many times longer does the detour take" ratio, not a distance proxy)
# ---------------------------------------------------------------------------

def compute_criticality(walking_edges_df):
    G_combined = G_sub.copy()
    for _, row in walking_edges_df.iterrows():
        u, v, w = row["trunk_stop"], row["sub_stop"], row["walk_sec"]
        if G_combined.has_edge(u, v):
            G_combined[u][v]["weight"] = min(G_combined[u][v]["weight"], w)
        else:
            G_combined.add_edge(u, v, weight=w)

    node_eff, node_deg = {n: 0.0 for n in G_trunk.nodes}, dict(G_trunk.degree())
    for u, v, d in G_trunk.edges(data=True):
        t_time = d["weight"]
        s_time = NO_BACKUP_PENALTY_SEC
        if u in G_combined and v in G_combined and nx.has_path(G_combined, u, v):
            s_time = nx.shortest_path_length(G_combined, u, v, weight="weight")
        e = t_time / (s_time + 1e-6)
        node_eff[u] += e
        node_eff[v] += e

    bet = nx.betweenness_centrality(G_trunk, weight="weight", normalized=True)
    clo = nx.closeness_centrality(G_trunk, distance="weight")

    df = pd.DataFrame({
        "node_id":     list(G_trunk.nodes),
        "stop_name":   [node_name_map.get(n, n) for n in G_trunk.nodes],
        "betweenness": [bet[n] for n in G_trunk.nodes],
        "closeness":   [clo[n] for n in G_trunk.nodes],
        "degree":      [node_deg[n] for n in G_trunk.nodes],
        "L_i":         [node_eff[n] for n in G_trunk.nodes],
    })
    df["CL_i"] = (df["betweenness"] * df["closeness"]) / (df["L_i"] + 1e-9)
    return df.sort_values("CL_i", ascending=False).reset_index(drop=True)

criticality_results = {}
for label, wt_sec in WALK_SCENARIOS.items():
    edges = build_walking_edges(wt_sec)
    df = compute_criticality(edges)
    criticality_results[label] = df
    print(f"\n--- {label} Scenario Top 5 ---")
    print(df[["stop_name", "CL_i", "L_i"]].head())

# ---------------------------------------------------------------------------
# RANK STABILITY ACROSS SCENARIOS
# ---------------------------------------------------------------------------

comparison = None
for label, df in criticality_results.items():
    ranked = df[["node_id", "stop_name", "CL_i"]].copy()
    ranked[f"rank_{label}"] = ranked["CL_i"].rank(ascending=False).astype(int)
    ranked = ranked.drop(columns=["CL_i"])
    comparison = ranked if comparison is None else comparison.merge(ranked, on=["node_id", "stop_name"], how="outer")

# Reference the first available scenario key for sorting
ref_label = sorted(WALK_SCENARIOS.keys())[0]
comparison = comparison.sort_values(f"rank_{ref_label}").reset_index(drop=True)
print("\n--- Rank Stability Across Scenarios ---")
print(comparison.head(15))

# ---------------------------------------------------------------------------
# VISUALIZATIONS
# ---------------------------------------------------------------------------

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G_trunk, k=0.5, iterations=50)
labels = {n: node_name_map.get(n, n) for n in G_trunk.nodes}
nx.draw(G_trunk, pos, labels=labels, node_size=700, node_color='skyblue',
         font_size=8, font_weight='bold', edge_color='gray')
plt.title("Silver Line Network (G_trunk) -- edge weights are scheduled travel time")
plt.tight_layout()
plt.savefig("silverline_network.png", dpi=150)
plt.show()

print(f"\n--- Correlation of CL_i with other centrality measures ({ref_label} scenario) ---")
df_ref = criticality_results[ref_label]
correlation_matrix = df_ref[['CL_i', 'betweenness', 'closeness', 'degree', 'L_i']].corr()
print(correlation_matrix['CL_i'].drop('CL_i'))

cl_i_comparison = pd.DataFrame({'node_id': df_ref['node_id'], 'stop_name': df_ref['stop_name']})
for label, df_res in criticality_results.items():
    cl_i_comparison = cl_i_comparison.merge(
        df_res[['node_id', 'CL_i']].rename(columns={'CL_i': f'CL_i_{label}'}),
        on='node_id', how='left'
    )

print("\n--- Correlation of CL_i scores across different walk scenarios ---")
print(cl_i_comparison.drop(columns=['node_id', 'stop_name']).corr())

# Dynamic scatter plot comparing first two scenarios if they exist
if len(WALK_SCENARIOS) >= 2:
    labels = list(WALK_SCENARIOS.keys())
    plt.figure(figsize=(8, 6))
    plt.scatter(cl_i_comparison[f'CL_i_{labels[0]}'], cl_i_comparison[f'CL_i_{labels[1]}'], alpha=0.7)
    plt.title(f'CL_i Scores: {labels[0]} vs {labels[1]} Walk Scenarios')
    plt.xlabel(f'CL_i ({labels[0]})')
    plt.ylabel(f'CL_i ({labels[1]})')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("cl_i_scatter.png", dpi=150)
    plt.show()

# ---------------------------------------------------------------------------
# SENSITIVITY: how does the "no backup" penalty change rankings?
# ---------------------------------------------------------------------------

def compute_crit_with_penalty(walking_edges_df, penalty_sec):
    G_combined = G_sub.copy()
    for _, row in walking_edges_df.iterrows():
        u, v, w = row["trunk_stop"], row["sub_stop"], row["walk_sec"]
        if G_combined.has_edge(u, v):
            G_combined[u][v]["weight"] = min(G_combined[u][v]["weight"], w)
        else:
            G_combined.add_edge(u, v, weight=w)

    node_eff = {n: 0.0 for n in G_trunk.nodes}
    for u, v, d in G_trunk.edges(data=True):
        t_time = d["weight"]
        s_time = penalty_sec
        if u in G_combined and v in G_combined and nx.has_path(G_combined, u, v):
            s_time = nx.shortest_path_length(G_combined, u, v, weight="weight")
        e = t_time / (s_time + 1e-6)
        node_eff[u] += e
        node_eff[v] += e

    bet = nx.betweenness_centrality(G_trunk, weight="weight", normalized=True)
    clo = nx.closeness_centrality(G_trunk, distance="weight")

    df = pd.DataFrame({
        "node_id":   list(G_trunk.nodes),
        "stop_name": [node_name_map.get(n, n) for n in G_trunk.nodes],
        "L_i":       [node_eff[n] for n in G_trunk.nodes],
    })
    bet_vals = pd.Series(bet)[df["node_id"]].values
    clo_vals = pd.Series(clo)[df["node_id"]].values
    df["CL_i"] = (bet_vals * clo_vals) / (df["L_i"] + 1e-9)
    return df

penalties_sec = [0, 1800, 10800]  # 0, 30 min, 3 hours
penalty_labels = ["Zero (0s)", "Moderate (30min)", "Extreme (3hr)"]
# Using the shortest scenario defined in WALK_SCENARIOS
walk_ref_sec = list(WALK_SCENARIOS.values())[0]
walk_ref_edges = build_walking_edges(walk_ref_sec)

sensitivity_df = pd.DataFrame({'node_id': list(G_trunk.nodes),
                                 'stop_name': [node_name_map.get(n, n) for n in G_trunk.nodes]})

for p, lbl in zip(penalties_sec, penalty_labels):
    res = compute_crit_with_penalty(walk_ref_edges, p)
    res = res.set_index('node_id').reindex(sensitivity_df['node_id'])
    sensitivity_df[f'CL_i_{lbl}'] = res['CL_i'].values
    sensitivity_df[f'Rank_{lbl}'] = res['CL_i'].rank(ascending=False).astype(int).values

print("\nComparison of Station Ranks under different 'no backup' penalties:")
print(sensitivity_df.sort_values(f'Rank_{penalty_labels[1]}').head(15))


In [ ]:
sentence = input("Please enter a sentence: ")

for char in sentence:
    print(char)